# SpiderNet data loading and MI-dimension-selection tutorial (Pancancer)

This notebook is a step-by-step tutorial for loading and preprocessing the Pancancer spatial transcriptomics data for SpiderNet, as well as running the MI-dimension-selection procedure on the processed data.

1. **Basic user inputs**
2. **Pancancer-specific SpiderNet input assumptions**
3. **Build the base config**
4. **Preview the LR-correlation density before choosing `lr_corr_threshold`**
5. **Run the full unified loader**
6. **Inspect outputs**
7. **Save Pancancer-specific batch metadata**
8. **Run MI dimension selection on the processed outputs**

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

# Cluster/project path. This makes imports robust when running with nbconvert/slurm.
PROJECT_ROOT = Path("/home/junjieta/SpiderNet")
for path_use in [PROJECT_ROOT, Path.cwd()]:
    path_str = str(path_use)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from SpiderNet.dataloading_unified import (
    prepare_processed_bundle_unified,
    preview_lr_corr_distribution,
)
from SpiderNet.utils import get_default_cellchat_db, get_default_scseqcomm_db


## Step 1. Basic user inputs

Edit the variables in this section first.

In [ ]:

# ---------------------------------------------------------------------
# Cluster paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("/home/junjieta/SpiderNet")
DATA_ROOT = PROJECT_ROOT / "Pancancer" / "Data"
RESULTS_ROOT = PROJECT_ROOT / "Pancancer" / "Results"

# Directly use the full/entire pan-cancer AnnData folder.
ADATA_FOLDER_NAME = "adata_entire"

# Processed objects generated from DATA_ROOT / ADATA_FOLDER_NAME.
# Keep this separate from model-training run directories.
OUTPUT_DIR = RESULTS_ROOT / "ProcessedData_entire"

SPECIES = "human"  ## Species used for ligand-receptor database loading. Options are "human" or "mouse".

SAMPLE_COL = "SampleID"  ## Pancancer batches are defined as <file_prefix>_subslice<subslice_id>, matching the original Pancancer loader.
CELL_CLASS_COL = "celltype_final"  ## The cell type annotation column in adata.obs.
SPATIAL_KEY = "spatial"  ## Spatial coordinates are read from adata.obsm[SPATIAL_KEY].
PYG_EXTRA_OBS_FIELDS = {}  ## Extra adata.obs fields to copy into each PyG sample object. Leave empty unless you want to propagate additional metadata.

N_HVG = 1000  ## Number of top highly variable genes used for SpiderNet model training.
N_HVG_LR = 2000  ## Number of top highly variable genes used when selecting ligand-receptor pairs from the training gene space.
NUM_NEIGHBORS = 5  ## Number of spatial neighbors. The Pancancer_modeltraining notebook uses 5 for this large dataset.

LR_CORR_THRESHOLD = None  ## Set this to the desired ligand-receptor correlation threshold after inspecting the preview plot in Step 4.

LR_LIST_PATH = None  ## Optional path to a predefined LR list (.pkl/.npy/.txt/.csv). If provided, the original Pancancer loader skips LR activation/correlation filtering.
GENE_LIST_PATH = None  ## Optional path to a predefined gene list (.pkl/.npy/.txt/.csv). If provided, the original Pancancer loader skips HVG selection.

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("ADATA directory:", DATA_ROOT / ADATA_FOLDER_NAME)
print("Processed output directory:", OUTPUT_DIR)


## Step 2. Pancancer-specific SpiderNet input

This notebook follows the Pancancer data convention used in the original
`dataloading_Pancancer` script and the `Pancancer_modeltraining` notebook:

- `.h5ad` files are read from `DATA_ROOT / "adata_entire"`
- expression is read directly from `X`
- zero-count cells are removed
- if `X` looks like raw integer counts, the loader runs `normalize_total(..., target_sum=1e4)` and `log1p`
- cell names are prefixed with the sample prefix extracted from the filename
- each cell is assigned a batch label  
  `SampleID = <file_prefix>_subslice<subslice_id>`
- spatial coordinates are stored in `obsm[SPATIAL_KEY]`
- cell types are defined by `obs["celltype_final"]`
- the Pancancer loader supports optional predefined LR lists and predefined gene lists
- the original Pancancer loader uses `lr_corr_threshold = 0.2` when LR filtering is enabled

In [ ]:
CELLCHAT_DB = get_default_cellchat_db(species=SPECIES)
SCSEQCOMM_DB = get_default_scseqcomm_db(species=SPECIES)

def normalize_optional_path_local(path_str):
    if path_str is None:
        return None
    path_str = str(path_str).strip()
    if path_str.lower() in {"", "none", "null", "nan"}:
        return None
    return path_str

def pancancer_file_prefix(context):
    return Path(context["file_name"]).stem.split("_")[0]

def pancancer_per_file_hook(adata, context):
    sample_prefix = pancancer_file_prefix(context)

    if sp.issparse(adata.X):
        adata.X = adata.X.astype(np.float32)
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    adata.obs_names = [f"{sample_prefix}_{cellname}" for cellname in adata.obs_names]

    if "subslice_id" not in adata.obs.columns:
        raise KeyError(
            "Pancancer preprocessing expects adata.obs['subslice_id'] so that "
            "SampleID can be constructed as <file_prefix>_subslice<subslice_id>."
        )

    adata.obs["source_file_prefix"] = sample_prefix
    adata.obs["SampleID"] = [
        f"{sample_prefix}_subslice{subslice_id}"
        for subslice_id in adata.obs["subslice_id"].astype(str)
    ]

    if SPATIAL_KEY not in adata.obsm:
        raise KeyError(
            f"Pancancer preprocessing expects adata.obsm[{SPATIAL_KEY!r}] to contain spatial coordinates."
        )

    adata.layers.clear()
    for key in ["X_pca", "X_umap"]:
        if key in adata.obsm:
            del adata.obsm[key]
    adata.obsm = {"spatial": np.asarray(adata.obsm[SPATIAL_KEY])}
    adata.varm.clear()
    adata.raw = None
    if adata.var.shape[1] > 0:
        adata.var = adata.var.iloc[:, []].copy()

    return adata

APPLY_HVG_SELECTION = normalize_optional_path_local(GENE_LIST_PATH) is None
APPLY_LR_CORR_FILTER = normalize_optional_path_local(LR_LIST_PATH) is None

DATA_REPRESENTATION_CONFIG = {
    "expression_source": {"kind": "X", "name": None},
    "normalize_strategy": "auto",
    "log1p": True,
    "remove_zero_count_cells": True,
    "spatial_source": {"kind": "obsm", "key": "spatial"},
    "apply_hvg_selection": APPLY_HVG_SELECTION,
}

## Step 3. Build the base config

This config contains the common loading settings for the Pancancer dataset.
`lr_corr_threshold` is intentionally injected at run time only when LR correlation
filtering is enabled.

In [ ]:
base_config = {
    "data_path_main": DATA_ROOT,
    "output_dir": OUTPUT_DIR,
    "adata_folder_name": ADATA_FOLDER_NAME,
    "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
    "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,

    "sample_col": SAMPLE_COL,
    "sample_name_obs_col": "source_file_prefix",
    "sample_attr_obs_col": SAMPLE_COL,
    "cell_class_col": CELL_CLASS_COL,
    "pyg_obs_fields": PYG_EXTRA_OBS_FIELDS,

    "n_hvg": N_HVG,
    "n_hvg_lr": N_HVG_LR,
    "num_neighbors": NUM_NEIGHBORS,

    "per_file_hook": pancancer_per_file_hook,
    "apply_lr_corr_filter": APPLY_LR_CORR_FILTER,
    "skip_lr_filter_when_predefined": True,

    **DATA_REPRESENTATION_CONFIG,
}

lr_list_path_use = normalize_optional_path_local(LR_LIST_PATH)
gene_list_path_use = normalize_optional_path_local(GENE_LIST_PATH)

if lr_list_path_use is not None:
    base_config["lr_list_path"] = lr_list_path_use

if gene_list_path_use is not None:
    base_config["gene_list_path"] = gene_list_path_use

base_config

## Step 4. Preview the LR-correlation density

Run this cell **before** the full loading step **only when no predefined LR list is supplied**.

If `LR_LIST_PATH` is provided, the original Pancancer loader skips LR activation/correlation
filtering, so this preview step is not needed.

In [ ]:
# if APPLY_LR_CORR_FILTER:
#     preview = preview_lr_corr_distribution(base_config, show_plot=True)
# 
#     if preview.get("plot_path") is not None:
#         print("Preview plot saved to:", preview["plot_path"])
# else:
#     preview = None
#     print(
#         "A predefined LR list was provided. To match the original Pancancer loader, "
#         "LR activation/correlation filtering is skipped, so the preview step is not needed."
#     )

## Step 5. Set `lr_corr_threshold` and run the full loader

After inspecting the density plot above, set `LR_CORR_THRESHOLD` to the desired value.
In the original PerturbFISH loader, the LR-correlation filter uses `0.5`, so that is a
reasonable default starting point here.


In [ ]:
LR_CORR_THRESHOLD = 0.2

In [ ]:
config = dict(base_config)

if APPLY_LR_CORR_FILTER:
    if LR_CORR_THRESHOLD is None:
        raise ValueError(
            "LR_CORR_THRESHOLD cannot be None when LR correlation filtering is enabled. "
            "Please inspect the preview plot and set a threshold, for example 0.2."
        )
    config["lr_corr_threshold"] = float(LR_CORR_THRESHOLD)

bundle = prepare_processed_bundle_unified(config)

## Step 6. Inspect the processed outputs

In [ ]:
print("Output dir:", bundle["output_dir"])
print("Number of batches:", len(bundle["batch_cell_unique"]))
print("Total cells:", bundle["adata"].n_obs)
print("All genes retained before training subset:", len(bundle["genenames"]))
print("Training genes:", len(bundle["genenames_train"]))
print("Retained LR pairs:", len(bundle["LR_list"]))
print("Number of cell types:", bundle["adata"].obs[CELL_CLASS_COL].nunique())
print("First 10 batch labels:", bundle["batch_cell_unique"][:10])

bundle["adata"]

## Step 7. Save Pancancer-specific batch metadata

This section writes one row per `SampleID` batch and saves it as
`metadata_sample.csv` in the processed output directory.

In [ ]:
obs = bundle["adata"].obs.copy()

metadata_sample = (
    obs.groupby(SAMPLE_COL, dropna=False)
      .agg(
          source_file_prefix=("source_file_prefix", "first"),
          subslice_id=("subslice_id", "first"),
          num_cells=(SAMPLE_COL, "size"),
          n_cell_types=(CELL_CLASS_COL, pd.Series.nunique),
      )
      .reset_index()
)

celltype_counts = pd.crosstab(obs[SAMPLE_COL], obs[CELL_CLASS_COL]).reset_index()
celltype_counts.columns = [SAMPLE_COL] + [f"n_{col}" for col in celltype_counts.columns[1:]]

metadata_sample = metadata_sample.merge(celltype_counts, on=SAMPLE_COL, how="left")
metadata_sample = metadata_sample.sort_values([ "source_file_prefix", "subslice_id", SAMPLE_COL ]).reset_index(drop=True)

metadata_sample_path = bundle["output_dir"] / "metadata_sample.csv"
metadata_sample.to_csv(metadata_sample_path, index=False)

print("Saved:", metadata_sample_path)
display(metadata_sample.head())

## Step 8. Run MI dimension selection on the processed outputs

This section reuses the processed files that were just written to `bundle["output_dir"]`.
It does **not** rerun data loading or preprocessing.

By default, the MI-dimension-selection thresholds are chosen automatically based on the
number of retained LR pairs. You can leave the optional overrides below as `None` unless
you want to tune the heuristic manually.

In [ ]:
from IPython.display import display
import pandas as pd

from SpiderNet.io import load_processed_data, get_spidernet_pyg_list_path, load_spidernet_pyg_list, spidernet_pyg_list_exists
from SpiderNet.MI_dimension_selection import run_mi_dimension_selection

# Optional advanced overrides for MI dimension selection.
# Leave these as None to use the default automatic heuristic.
MI_DIM_LR_SPEARCOR_THRESHOLD = None
MI_DIM_MIN_CLIQUE_SIZE = None
MI_DIM_JACCARD_THRESHOLD = None
MI_DIM_SHOW_HEATMAP = True

processed = load_processed_data(bundle["output_dir"])

print("Processed directory:", bundle["output_dir"])
print("Number of batches:", len(processed.spidernet_data))
print("Number of retained LR pairs:", len(processed.lr_list))
print("Number of training genes:", len(processed.genenames_train))

mi_dim_results = run_mi_dimension_selection(
    processed=processed,
    lr_list=processed.lr_list,
    output_dir=bundle["output_dir"],
    lr_spearcor_threshold=MI_DIM_LR_SPEARCOR_THRESHOLD,
    min_clique_size=MI_DIM_MIN_CLIQUE_SIZE,
    jaccard_thr=MI_DIM_JACCARD_THRESHOLD,
    show=MI_DIM_SHOW_HEATMAP,
)

mi_dim_summary_df = pd.DataFrame(
    [
        {
            "recommended_dim_envir": mi_dim_results["recommended_dim_envir"],
            "num_merged_subsets": mi_dim_results["num_merged_subsets"],
            "subset_sizes": mi_dim_results["subset_sizes"],
            "num_lr_pairs": mi_dim_results["num_lr_pairs"],
            "num_graph_edges": mi_dim_results["num_graph_edges"],
            "num_maximal_cliques_filtered": mi_dim_results["num_maximal_cliques_filtered"],
            "effective_min_clique_size": mi_dim_results["effective_min_clique_size"],
            "num_unassigned_lr_pairs": mi_dim_results["num_unassigned_lr_pairs"],
            "lr_spearcor_threshold": mi_dim_results["lr_spearcor_threshold"],
            "jaccard_thr": mi_dim_results["jaccard_thr"],
            "warning": mi_dim_results["warning"],
        }
    ]
)

display(mi_dim_summary_df)

print(f"Recommended dim_envir: {mi_dim_results['recommended_dim_envir']}")
print(f"Heatmap saved to: {mi_dim_results['heatmap_path']}")
print(f"Summary JSON saved to: {mi_dim_results['summary_path']}")
print(f"Subset summary CSV saved to: {mi_dim_results['subset_summary_path']}")